# 03 · Forecasting monthly collisions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam843/uk-road-accident-analytics/blob/main/notebooks/03_forecasting.ipynb)

**Series:** reported road collisions per month, GB, 2005-01 to 2019-12 (180 months, no gaps,
month-start frequency).

**Splits (locked):** training 2005-01..2016-12; test 2017-01..2019-12. The test period is used once,
at the end.

**Models**
| Model | What it does |
|---|---|
| Seasonal naive | next January = last January, and so on. The baseline every other model must beat. |
| ETS | Holt-Winters exponential smoothing: additive damped trend, multiplicative seasonality (statsmodels). |
| SARIMA | `auto_arima` (pmdarima) chooses the order **once, on the training data**. The order is then fixed and reported, and only its coefficients are re-estimated at each origin. |

**Evaluation**
1. Rolling-origin backtest inside the training period: origins every 12 months from 2012-12
   (2012-12, 2013-12, 2014-12, 2015-12), horizon 12. Each fit sees only data up to its origin.
2. Final test: fit on 2005-01..2016-12, forecast 2017-01..2019-12 (36 months).
3. Replica check: fit on 2005-01..2014-12, forecast the 2015 annual total. Compared with the
   official RAS0101 figure and with seasonal naive.
4. Residual diagnostics for SARIMA (Ljung-Box, ACF).

Caveat: the SARIMA order is chosen with data up to 2016-12, so the earlier backtest origins
benefit slightly from an order chosen with later data. Coefficients are still re-fitted using
data up to each origin only.

Results: **TBD (run notebooks/03_forecasting.ipynb)**.

In [ ]:
# Setup: works from the repo root, from notebooks/, or on Google Colab.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("src/config.py").exists():
    if not Path("uk-road-accident-analytics").exists():
        !git clone -q https://github.com/maryam843/uk-road-accident-analytics.git
    %cd uk-road-accident-analytics
    %pip install -q -r requirements.txt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd

from src import clean, config, evaluate

config.setup_logging(logging.INFO)
evaluate.apply_style()
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

DATA_DIR = config.get_data_dir()
RAW, PROCESSED = DATA_DIR / "raw", DATA_DIR / "processed"
if not (PROCESSED / config.COLLISIONS_PARQUET).exists():
    # Fresh runtime (e.g. Colab): download (several GB) and clean. Takes a while, once.
    from src import download

    download.main([])
    clean.run()
print("DATA_DIR =", DATA_DIR)

In [ ]:
import warnings

import numpy as np
import pmdarima as pm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

monthly = pd.read_parquet(PROCESSED / config.MONTHLY_PARQUET)["collisions"].astype(float)
monthly.index = pd.DatetimeIndex(monthly.index, freq="MS")
assert len(monthly) == 180 and monthly.index.freqstr == "MS" and monthly.notna().all()
assert (monthly > 0).all(), "a month with zero collisions would make MAPE undefined"

train = monthly.loc[:config.FORECAST_TRAIN_END]
test = monthly.loc[config.FORECAST_TEST_START:config.FORECAST_TEST_END]
print(f"train: {train.index[0]:%Y-%m}..{train.index[-1]:%Y-%m} ({len(train)} months); "
      f"test: {test.index[0]:%Y-%m}..{test.index[-1]:%Y-%m} ({len(test)} months)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train.index, train / 1000, label="Training (2005-2016)")
ax.plot(test.index, test / 1000, label="Test (2017-2019)")
ax.set_title("Reported road collisions per month, GB")
ax.set_ylabel("Collisions (thousands per month)")
ax.legend()
evaluate.save_fig(fig, "03_monthly_series")
plt.show()

## Choose the SARIMA order on the training data

In [ ]:
auto = pm.auto_arima(train, seasonal=True, m=12, stepwise=True, information_criterion="aic",
                     suppress_warnings=True, error_action="ignore", random_state=config.RANDOM_STATE)
ORDER, SEASONAL_ORDER, WITH_INTERCEPT = auto.order, auto.seasonal_order, auto.with_intercept
print(f"Chosen on 2005-01..2016-12: SARIMA{ORDER}x{SEASONAL_ORDER}, intercept={WITH_INTERCEPT}, "
      f"AIC={auto.aic():,.1f}")

In [ ]:
def fc_seasonal_naive(y, h):
    return evaluate.seasonal_naive(y, h).to_numpy()


def fc_ets(y, h):
    fit = ExponentialSmoothing(y, trend="add", damped_trend=True, seasonal="mul",
                               seasonal_periods=12, initialization_method="estimated").fit()
    return np.asarray(fit.forecast(h))


def fit_sarima(y):
    return pm.ARIMA(order=ORDER, seasonal_order=SEASONAL_ORDER, with_intercept=WITH_INTERCEPT,
                    suppress_warnings=True).fit(y)


def fc_sarima(y, h):
    return np.asarray(fit_sarima(y).predict(n_periods=h))


MODELS = {"Seasonal naive": fc_seasonal_naive, "ETS": fc_ets, "SARIMA": fc_sarima}

## Rolling-origin backtest (inside the training period)

In [ ]:
origins = list(pd.date_range(config.BACKTEST_FIRST_ORIGIN, train.index[-1] - pd.DateOffset(months=12),
                             freq="12MS"))
print("origins:", [f"{o:%Y-%m}" for o in origins])
backtest = evaluate.rolling_origin_backtest(train, origins, config.BACKTEST_HORIZON, MODELS)
per_origin = backtest.groupby(["origin", "model"]).apply(
    lambda g: evaluate.mae(g["actual"], g["forecast"]), include_groups=False).unstack("model")
print("MAE per origin (collisions per month):")
display(per_origin.style.format("{:,.0f}"))

## Final test: 2017-01..2019-12

In [ ]:
test_fc = pd.DataFrame({name: fn(train, len(test)) for name, fn in MODELS.items()}, index=test.index)
metrics = pd.concat(
    {
        "Backtest 2013-2016 (4 origins × 12 months)": evaluate.summarise_forecasts(backtest),
        "Test 2017-2019 (36 months)": pd.DataFrame(
            {m: evaluate.forecast_metrics(test, test_fc[m]) for m in MODELS}).T,
    },
    axis=1,
)
display(metrics.style.format("{:,.2f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
history = monthly.loc["2014-01":config.FORECAST_TRAIN_END]
ax.plot(history.index, history / 1000, color="#52514e", linewidth=1.5, label="Actual (training)")
ax.plot(test.index, test / 1000, color="#0b0b0b", linewidth=2.2, label="Actual (test)")
for name in MODELS:
    ax.plot(test.index, test_fc[name] / 1000, linestyle="--", label=name)
ax.axvline(test.index[0], color="#52514e", linewidth=0.8)
ax.set_title("Test-period forecasts, fitted on 2005-2016")
ax.set_ylabel("Collisions (thousands per month)")
ax.legend(ncols=3, fontsize=8)
evaluate.save_fig(fig, "03_test_forecasts")
plt.show()

## Replica check: forecast the 2015 total from data up to 2014

In [ ]:
train_2014 = monthly.loc[:"2014-12"]
actual_2015 = monthly.loc["2015"].sum()
official = clean.load_ras0101(RAW / config.RAS0101_RAW_FILE)
official_2015 = float(official.loc[2015, "collisions"])
replica = pd.DataFrame({name: {"forecast_2015_total": fn(train_2014, 12).sum()} for name, fn in MODELS.items()}).T
replica["pipeline_actual_2015"] = actual_2015
replica["ras0101_official_2015"] = official_2015
replica["error_vs_official"] = replica["forecast_2015_total"] - official_2015
replica["pct_error_vs_official"] = 100 * replica["error_vs_official"] / official_2015
display(replica.style.format("{:,.1f}"))

## SARIMA residual diagnostics (fit on 2005-2016)

In [ ]:
sarima_train = fit_sarima(train)
burn_in = ORDER[1] + 12 * SEASONAL_ORDER[1]
resid = pd.Series(sarima_train.resid(), index=train.index).iloc[burn_in:]
n_params = ORDER[0] + ORDER[2] + SEASONAL_ORDER[0] + SEASONAL_ORDER[2]
lb = acorr_ljungbox(resid, lags=[12, 24], model_df=n_params)
display(lb.style.format({"lb_stat": "{:.2f}", "lb_pvalue": "{:.3f}"}))
print("Ljung-Box H0: residuals are uncorrelated up to the given lag; "
      "a small p-value means structure is left in the residuals.")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(resid.index, resid)
axes[0].axhline(0, color="#52514e", linewidth=0.8)
axes[0].set_title("SARIMA residuals")
axes[0].set_ylabel("Collisions per month")
plot_acf(resid, lags=36, ax=axes[1], title="Residual ACF (36 lags)")
fig.tight_layout()
evaluate.save_fig(fig, "03_sarima_residuals")
plt.show()

## Verdict

In [ ]:
for period in metrics.columns.levels[0]:
    mae_naive = metrics.loc["Seasonal naive", (period, "MAE")]
    mae_sarima = metrics.loc["SARIMA", (period, "MAE")]
    best = metrics[(period, "MAE")].idxmin()
    if mae_sarima < mae_naive:
        verdict = f"SARIMA beats seasonal naive (MAE {mae_sarima:,.0f} vs {mae_naive:,.0f})"
    else:
        verdict = (f"SARIMA does NOT beat seasonal naive (MAE {mae_sarima:,.0f} vs "
                   f"{mae_naive:,.0f}); the simpler baseline is at least as good")
    print(f"{period}: {verdict}. Lowest MAE: {best}.")

Interpretation: **TBD (run notebooks/03_forecasting.ipynb)**.

Notes for reading the numbers:
- Monthly counts are not adjusted for the number of days in a month. All models see the same
  series, so the comparison is fair, but part of the "seasonality" is calendar length.
- A forecast of reported collisions inherits any change in police reporting practice. The
  severity reporting change affects severity, not the all-collision count used here.